# Ücretsiz AI Video Generator — Google Colab
**Hedef:** 15 sahne × 12 saniye = **tam 3 dakika (180 saniye)**, Türkçe seslendirmeli 1280×720 MP4.

**Kullanım:** Google Colab'da **Çalışma Zamanı → Tümünü çalıştır**. Yerel programa ihtiyaç yoktur. Son hücre videoyu tarayıcı içinde oynatır; indirmek zorunlu değildir.

- Varsayılan: ücretsiz *edge-tts* Türkçe yapay zekâ sesi + Python ile çizilmiş sinematik arka plan + sahne başlıkları + hafif kamera yakınlaşması.
- İsteğe bağlı: `USE_AI_IMAGES = True` seçilirse GPU varsa açık kaynak SDXL-Turbo ile görsel üretilmesi denenir. Bu adım internet, GPU, model erişimi ve lisans koşullarına bağlıdır; başarısız olursa çizim arka planlarına döner.
- **Önemli:** Görsellerin hafif yakınlaşması gerçek hareketli karakter animasyonu değildir. Bu defterin temel modu bir *seslendirmeli hareketli sunum* üretir.
- Colab ücretsiz GPU ve çalışma süresi **garanti edilmez**. Kullanılan ücretsiz üçüncü taraf ses hizmeti zaman zaman erişilemeyebilir. Ticari kullanım öncesi ses/model lisanslarını ayrıca kontrol edin.

In [ ]:
import os, sys, subprocess
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "edge-tts", "Pillow", "numpy"], check=True)
subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, check=True)
OUT = Path("/content/ai_video_generator")
OUT.mkdir(parents=True, exist_ok=True)
print("Çalışma klasörü:", OUT)


In [ ]:
# İstediğiniz konuyu, ses tipini ve sahne metinlerini burada değiştirin.
VIDEO_TITLE = "Yapay Zekâ ile Video Üretimi"
VOICE = "tr-TR-AhmetNeural"  # Alternatif: tr-TR-EmelNeural
VOICE_RATE = "+0%"
USE_AI_IMAGES = False  # True: ücretsiz Colab GPU üzerinde SDXL-Turbo denenir.
SCENE_SECONDS = 12
FPS = 24
WIDTH, HEIGHT = 1280, 720

SCENES = [
  {
    "title": "Gelecek bugün başlıyor",
    "narration": "Bir sabah uyandığını düşün. Telefonun yalnızca alarm çalmıyor. Gününü planlıyor, haberleri özetliyor ve sana gerçekten önemli olan işlerini hatırlatıyor.",
    "prompt": "futuristic morning bedroom soft sunlight holographic calendar cinematic"
  },
  {
    "title": "Yeni bir yardımcı",
    "narration": "Yapay zekâ artık sadece sorulara cevap veren bir araç değil. Yazıları düzenleyebiliyor, fikir üretebiliyor ve tekrar eden işleri daha kısa sürede tamamlamana yardımcı oluyor.",
    "prompt": "helpful friendly artificial intelligence assistant at desk photorealistic"
  },
  {
    "title": "Bir fikirle başla",
    "narration": "Örneğin bir YouTube videosu hazırlamak istiyorsun. Önce konuyu belirliyorsun. Sonra yapay zekâdan izleyiciyi meraklandıracak bir hikâye taslağı oluşturmasını istiyorsun.",
    "prompt": "creator developing video storyboard on digital desk"
  },
  {
    "title": "Hikâyeyi sahnelere ayır",
    "narration": "İyi bir video tek bir uzun görüntü değildir. Kısa sahneler, güçlü açılışlar ve anlaşılır geçişler izleyen kişinin dikkatini canlı tutar.",
    "prompt": "cinematic storyboard sheets and concept art"
  },
  {
    "title": "Görselleri üret",
    "narration": "Metinden görsel üreten modeller, hayalindeki ortamları canlandırabilir. Mekânı, ışığı, karakterleri ve kamera açısını tarif ederek çok farklı sahneler deneyebilirsin.",
    "prompt": "magical digital landscape cinematic concept art"
  },
  {
    "title": "Hareket duygusu kat",
    "narration": "Sabit bir görsel bile yavaş yakınlaşma, kaydırma ve dikkatli geçişlerle hareket hissi verebilir. Gerçek video üretimi ise daha fazla işlem gücü gerektirir.",
    "prompt": "cinematic camera tracking across beautiful illustrated landscape"
  },
  {
    "title": "Doğal bir ses seç",
    "narration": "Seslendirme videonun ruhunu belirler. Anlaşılır, doğru vurgulu ve rahat dinlenen bir Türkçe ses, en etkileyici görselden bile daha önemli olabilir.",
    "prompt": "professional studio microphone dramatic soft lighting"
  },
  {
    "title": "Metni konuşmaya dönüştür",
    "narration": "Bugün bazı ses araçları metni otomatik okuyabiliyor. Bu yöntemle mikrofon kullanmadan bir anlatım oluşturabilir ve ses dosyasını sahnelerin üzerine yerleştirebilirsin.",
    "prompt": "audio waveform illuminated in recording studio"
  },
  {
    "title": "Ritmi koru",
    "narration": "Her sahne için doğru süreyi seçmek gerekir. Çok hızlı anlatım yorucu, çok uzun sessizlik ise sıkıcı olabilir. Görüntüler anlatılan düşünceyle uyumlu ilerlemeli.",
    "prompt": "cinematic editing timeline on laptop screen"
  },
  {
    "title": "Altyazıyı unutma",
    "narration": "İzleyicilerin bir bölümü videoları sessiz izler. Okunaklı başlıklar ve doğru zamanlanmış altyazılar mesajın daha fazla kişiye ulaşmasına yardımcı olur.",
    "prompt": "beautiful legible captions in cinematic frame"
  },
  {
    "title": "Müzik ve telif",
    "narration": "Arka plan müziği kullanacaksan kullanım hakkını kontrol etmelisin. Ücretsiz görünen her ses ticari kullanım için serbest olmayabilir. Güvenilir lisansları tercih et.",
    "prompt": "cinematic musical notes studio headphones"
  },
  {
    "title": "Ücretsiz araçlarla dene",
    "narration": "Google Colab üzerinde Python ve FFmpeg gibi araçlarla basit bir video düzeni kurulabilir. Ücretsiz kaynakların süresi ve kullanılabilirliği her zaman garanti değildir.",
    "prompt": "laptop cloud computing visual abstraction"
  },
  {
    "title": "Kaliteyi gözden geçir",
    "narration": "Video hazır olduğunda telaffuzu, kesilen cümleleri, görüntü bütünlüğünü ve yazıların okunurluğunu kontrol et. Küçük düzenlemeler sonucun kalitesini belirgin biçimde artırır.",
    "prompt": "film editor checking cinematic footage"
  },
  {
    "title": "Bir kez değil geliştirerek",
    "narration": "İlk üretimin kusursuz olması gerekmez. Kısa denemelerle sahne düzenini, ses hızını ve görsel anlatımı geliştirerek kendine ait bir video tarzı oluşturabilirsin.",
    "prompt": "creative workspace with cinematic sketches"
  },
  {
    "title": "Sıradaki video senin",
    "narration": "Şimdi elinde bir başlangıç var. Bir konu seç, metni kişiselleştir ve sahneleri kendi hikâyene uyarla. Üretmeye başladığında yeni fikirler kendiliğinden gelecektir.",
    "prompt": "sunrise on cinematic horizon hopeful future"
  }
]
assert len(SCENES) == 15, "Tam 3 dakika için 15 sahne gerekir."
print(f"{len(SCENES)} sahne, toplam {len(SCENES)*SCENE_SECONDS} saniye.")


In [ ]:
import asyncio, shutil
import edge_tts

async def synthesize_all():
    for i, scene in enumerate(SCENES, 1):
        p = OUT / f"ses_{i:02}.mp3"
        if p.exists() and p.stat().st_size > 1000:
            continue
        for attempt in range(3):
            try:
                await edge_tts.Communicate(scene["narration"], VOICE, rate=VOICE_RATE).save(str(p))
                print(f"Ses {i:02}/{len(SCENES)} oluşturuldu")
                break
            except Exception as ex:
                if attempt == 2:
                    raise RuntimeError(f"Türkçe ses servisi kullanılamadı, sahne {i}: {ex}")
                await asyncio.sleep(2 * (attempt + 1))
        await asyncio.sleep(0.35)

await synthesize_all()

def duration_seconds(path):
    p = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
        capture_output=True, text=True, check=True
    )
    return float(p.stdout.strip())

# Uzun cümleleri sahneye sığdırmak için sesi gerektiği kadar hızlandır.
for i in range(1, len(SCENES)+1):
    original = OUT / f"ses_{i:02}.mp3"
    playable = OUT / f"ses_final_{i:02}.mp3"
    d = duration_seconds(original)
    if d > SCENE_SECONDS - 0.4:
        factor = max(1.0, d / (SCENE_SECONDS - 0.5))
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(original),
             "-filter:a", f"atempo={factor:.4f}", str(playable)], check=True)
    else:
        shutil.copyfile(original, playable)
print("Türkçe seslendirmeler tamamlandı.")


In [ ]:
import math, random
from PIL import Image, ImageDraw, ImageFont
import numpy as np

FONT_BOLD = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
FONT_BODY = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

def wrap_words(draw, text, font, max_width):
    lines, current = [], ""
    for word in text.split():
        attempt = (current + " " + word).strip()
        if draw.textbbox((0, 0), attempt, font=font)[2] > max_width and current:
            lines.append(current)
            current = word
        else:
            current = attempt
    if current:
        lines.append(current)
    return lines

def make_background(index):
    rng = random.Random(221 + index)
    top = np.array((14 + index % 4 * 5, 16, 43 + index % 3 * 7), dtype=np.float32)
    bottom = np.array((10, 43 + index % 4 * 8, 65 + index % 4 * 7), dtype=np.float32)
    y = np.linspace(0, 1, HEIGHT)[:, None, None]
    x = np.linspace(0, 1, WIDTH)[None, :, None]
    grad = np.broadcast_to(top * (1-y) + bottom*y, (HEIGHT, WIDTH, 3)).copy()
    light = 18 * np.exp(-((x-0.78)**2 + (y-0.42)**2)/0.09)
    grad += light
    im = Image.fromarray(np.uint8(np.clip(grad, 0, 255)), "RGB")
    art = ImageDraw.Draw(im, "RGBA")
    cx, cy = int(WIDTH*0.73), int(HEIGHT*0.42)
    for k in range(15, 0, -1):
        r = k * 26
        art.ellipse((cx-r,cy-r,cx+r,cy+r), outline=(72,196,228,16+k*2), width=3)
    for n in range(55):
        px,py = rng.randrange(WIDTH),rng.randrange(HEIGHT)
        rad = rng.randint(1,4)
        art.ellipse((px-rad,py-rad,px+rad,py+rad),fill=(242,205,140,rng.randint(35,145)))
    art.polygon([(0,HEIGHT-60),(260,HEIGHT-130),(450,HEIGHT-70),(790,HEIGHT-140),(WIDTH,HEIGHT-75),(WIDTH,HEIGHT),(0,HEIGHT)],fill=(5,13,32,125))
    return im

def label_scene(im, index, scene):
    d = ImageDraw.Draw(im, "RGBA")
    bold = ImageFont.truetype(FONT_BOLD, 58)
    subtitle_font = ImageFont.truetype(FONT_BODY, 25)
    small = ImageFont.truetype(FONT_BOLD, 22)
    d.rounded_rectangle((52,55,365,105),radius=21,fill=(21,38,61,210),outline=(76,191,214,130),width=2)
    d.text((75,68), f"SAHNE {index:02d} / 15",font=small,fill=(182,234,239))
    headings = wrap_words(d,scene["title"],bold,1000)
    for j,line in enumerate(headings):
        d.text((67,210+j*74),line,font=bold,fill=(255,255,255),stroke_width=2,stroke_fill=(8,23,45))
    # Her sahnenin tüm anlatımı okunabilir bir altyazı kartında gösterilir.
    lines = wrap_words(d,scene["narration"],subtitle_font,1120)
    h = max(115, 30+len(lines)*40)
    y0 = HEIGHT-h-30
    d.rounded_rectangle((48,y0,WIDTH-48,y0+h),radius=24,fill=(7,14,33,222),outline=(100,203,221,75),width=2)
    for j,line in enumerate(lines):
        d.text((77,y0+20+j*40),line,font=subtitle_font,fill=(239,246,254))
    return im

ai_pictures = {}
if USE_AI_IMAGES:
    try:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("Colab GPU bulunamadı.")
        subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                        "diffusers", "transformers", "accelerate", "safetensors"],check=True)
        from diffusers import AutoPipelineForText2Image
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
        ).to("cuda")
        for i,scene in enumerate(SCENES,1):
            try:
                result = pipe(prompt=scene["prompt"] + ", landscape 16:9, no text, no logos",
                              num_inference_steps=2, guidance_scale=0.0,
                              width=768, height=448).images[0]
                ai_pictures[i] = result.resize((WIDTH, HEIGHT))
                print(f"AI görsel {i}/15")
            except Exception as ex:
                print(f"Sahne {i} çizim moduna alındı: {ex}")
        del pipe
        torch.cuda.empty_cache()
    except Exception as ex:
        print("AI görsel modeli kullanılamadı; ücretsiz çizim arka planına geçiliyor:",ex)

for i,scene in enumerate(SCENES,1):
    img = ai_pictures.get(i,make_background(i))
    label_scene(img,i,scene).save(OUT / f"sahne_{i:02d}.png")
print("Tüm görseller hazır.")


In [ ]:
from IPython.display import display, Video

clip_paths = []
for i in range(1, len(SCENES)+1):
    image_path = OUT / f"sahne_{i:02d}.png"
    audio_path = OUT / f"ses_final_{i:02d}.mp3"
    clip_path = OUT / f"klip_{i:02d}.mp4"
    vf = (f"zoompan=z='min(zoom+0.00055,1.13)':"
          f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':"
          f"d={FPS}:s={WIDTH}x{HEIGHT}:fps={FPS},format=yuv420p")
    cmd = [
        "ffmpeg", "-y", "-loglevel", "error",
        "-loop", "1", "-framerate", "1", "-i", str(image_path),
        "-i", str(audio_path),
        "-filter_complex", f"[0:v]{vf}[v];[1:a]apad,atrim=0:{SCENE_SECONDS},asetpts=N/SR/TB[a]",
        "-map", "[v]", "-map", "[a]",
        "-t", str(SCENE_SECONDS), "-r", str(FPS),
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "26",
        "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k",
        "-ar", "44100", "-ac", "2", str(clip_path)
    ]
    subprocess.run(cmd,check=True)
    clip_paths.append(clip_path)
    print(f"Klip {i}/{len(SCENES)} hazır")

manifest = OUT / "clips.txt"
manifest.write_text("".join(f"file '{str(p)}'\n" for p in clip_paths), encoding="utf-8")
final_path = OUT / "AI_Video_3_Dakika.mp4"
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-f", "concat", "-safe", "0", "-i", str(manifest),
    "-c", "copy", "-movflags", "+faststart", str(final_path)
],check=True)

duration = duration_seconds(final_path)
print(f"Video: {final_path} | Süre: {duration:.2f} saniye | Boyut: {final_path.stat().st_size/1e6:.1f} MB")
assert abs(duration - 180) < 0.6, "Video süresi 180 saniye değil."


In [ ]:
# Video, Google Colab içinde oynatılır. İndirme gerektirmez.
display(Video(str(final_path), embed=True, width=800))
print("Dosya Colab oturumu açık olduğu sürece", final_path, "konumundadır.")
# İsteğe bağlı yerel indirme: aşağıdaki iki satırın başındaki # işaretini kaldırın.
# from google.colab import files
# files.download(str(final_path))
